# Решения: kmeans/dbscan

**Для преподавателя.** Эталон к `lesson.ipynb` и `homework.ipynb`. Не показывать ученикам до сдачи.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


def _find(name: str) -> Path:
    for p in (Path(name), Path(f'../../data/{name}'), Path(f'../data/{name}')):
        if p.exists():
            return p.resolve()
    raise FileNotFoundError(f'{name} не найден рядом с ноутбуком')


CSV_PATH = _find('orders_slim.csv')
df = pd.read_csv(
    CSV_PATH,
    parse_dates=['order_purchase_timestamp', 'order_estimated_delivery_date', 'order_delivered_customer_date'],
)

from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler


In [ ]:
features = ['delivery_days', 'freight_value', 'delay_days']
X = df[features].copy()
scaler = StandardScaler()
Xs = scaler.fit_transform(X)
km = KMeans(n_clusters=3, random_state=53, n_init=10)
labels_km = km.fit_predict(Xs)
db = DBSCAN(eps=0.9, min_samples=4)
labels_db = db.fit_predict(Xs)
tmp = df.copy()
tmp['cluster_km'] = labels_km
tmp['cluster_db'] = labels_db
profile = tmp.groupby('cluster_km')[features + ['is_late']].mean().round(2)
rows = []
for k in (2, 3, 4, 5):
    m = KMeans(n_clusters=k, random_state=53, n_init=10)
    lbl = m.fit_predict(Xs)
    rows.append({'k': k, 'inertia': float(m.inertia_), 'n_clusters': int(pd.Series(lbl).nunique())})
k_table = pd.DataFrame(rows)
CLUSTER_NOTE = (
    'KMeans удобен, когда хотим фиксированное число сегментов. '
    'DBSCAN лучше ловит шум и аномалии, но чувствителен к eps/min_samples и масштабу признаков.'
)
print(pd.Series(labels_km).value_counts())
print(pd.Series(labels_db).value_counts())
print(profile)
print(k_table)
print(CLUSTER_NOTE)